In [3]:
from huggingface_hub import list_repo_files

repo_id = "labiaufba/PublicTransportationSunt"

print("Archivos disponibles en el dataset:")
od_files = []
for f in list_repo_files(repo_id, repo_type="dataset"):
    if f.startswith("OD/od-"):
        od_files.append(f)

od_files.sort()
print(f"Total archivos OD: {len(od_files)}")
print(f"Primero: {od_files[0]}")
print(f"Último:  {od_files[-1]}")

# Extraer los meses únicos disponibles
import re
months_available = set()
for f in od_files:
    match = re.search(r'od-(\d{4})-(\d{2})-\d{2}\.parquet', f)
    if match:
        months_available.add((int(match.group(1)), int(match.group(2))))

months_available = sorted(months_available)
print(f"\nMeses disponibles: {months_available}")
print(f"\nPegar esto en MONTHS_TO_DOWNLOAD:")
print(f"MONTHS_TO_DOWNLOAD = {months_available}")

Archivos disponibles en el dataset:
Total archivos OD: 183
Primero: OD/od-2024-03-01.parquet
Último:  OD/od-2025-03-14.parquet

Meses disponibles: [(2024, 3), (2024, 4), (2024, 5), (2024, 6), (2024, 7), (2024, 8), (2024, 9), (2024, 10), (2024, 11), (2024, 12), (2025, 1), (2025, 2), (2025, 3)]

Pegar esto en MONTHS_TO_DOWNLOAD:
MONTHS_TO_DOWNLOAD = [(2024, 3), (2024, 4), (2024, 5), (2024, 6), (2024, 7), (2024, 8), (2024, 9), (2024, 10), (2024, 11), (2024, 12), (2025, 1), (2025, 2), (2025, 3)]


# SUNT OD Dataloader
Downloads, validates, and saves the SUNT origin-destination dataset from hugging face.

In [2]:
# --- imports ---
from huggingface_hub import hf_hub_download
import pandas as pd
from datetime import datetime
import calendar
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [10]:
# --- configuration ---

# months to download: list of (year, month) tuples
# example: download march and april 2024
# MONTHS_TO_DOWNLOAD = [(2024, 3), (2024, 4)]
# to download just one month:
# MONTHS_TO_DOWNLOAD = [(2024, 3)]

MONTHS_TO_DOWNLOAD = [(2024, 3), (2024, 6), (2024, 9), (2024, 12)]  # 4 months

# where to save the processed data inside your drive
# SAVE_PATH = "./data"

SAVE_PATH = "../data" # changed now to save outside the repo

# standard bus capacity assumed for occupancy calculation
BUS_CAPACITY = 80

# show download progress bar
SHOW_PROGRESS = True

# validate data after downloading
VALIDATE_DATA = True

In [11]:
# --- download one month from hugging face ---

def download_month(year, month):
    """
    downloads all available days for a given month from the sunt od dataset.
    returns a concatenated dataframe, or none if nothing could be downloaded.
    """
    num_days = calendar.monthrange(year, month)[1]
    month_name = calendar.month_name[month]

    print(f"\n{'='*60}")
    print(f"Downloading sunt od - {month_name} {year}")
    print(f"{'='*60}")
    print(f"Days in month: {num_days}")

    dfs = []
    failed_days = []

    day_range = tqdm(range(1, num_days + 1), desc="downloading", unit="day") if SHOW_PROGRESS else range(1, num_days + 1)

    for day in day_range:
        date_str = datetime(year, month, day).strftime("%Y-%m-%d")
        filename = f"OD/od-{date_str}.parquet"

        try:
            file_path = hf_hub_download(
                repo_id="labiaufba/PublicTransportationSunt",
                filename=filename,
                repo_type="dataset"
            )
            df_day = pd.read_parquet(file_path)

            if len(df_day) == 0:
                failed_days.append(date_str)
                continue

            dfs.append(df_day)

        except Exception:
            failed_days.append(date_str)
            continue

    print(f"\nDays downloaded successfully: {num_days - len(failed_days)}/{num_days}")

    if not dfs:
        print("error: could not download any day for this month")
        return None

    df = pd.concat(dfs, ignore_index=True)
    print(f"total records: {len(df):,}")
    print(f"memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} mb")

    return df

In [12]:
# --- validate downloaded data ---

def validate_data(df):
    """
    checks that all critical columns are present and shows basic stats.
    returns true if data is valid, false otherwise.
    """
    print(f"\n{'='*60}")
    print("Data validation")
    print(f"{'='*60}")

    # columns we need for the occupancy model
    critical_columns = {
      'loading':          'passenger count on bus (critical)',
      'n-boardings':      'number of boardings at stop',
      'n-alighting':      'number of alightings at stop',
      'route_short_name': 'bus line',
      'stop_id':          'stop identifier',
      'direction_id':     'trip direction',
      'pt_sequence':      'stop sequence within trip',
      'stop_time':        'timestamp',
    }

    missing = [col for col in critical_columns if col not in df.columns]

    for col, desc in critical_columns.items():
        status = "ok" if col in df.columns else "MISSING"
        print(f"  {status:8s} {col:20s} - {desc}")

    if missing:
        print(f"\nwarning: {len(missing)} critical columns are missing")
        return False

    # loading stats
    print(f"\nloading (passenger count) stats:")
    print(f"valid values : {df['loading'].notna().sum():,}")
    print(f"nan values : {df['loading'].isna().sum():,} ({df['loading'].isna().mean()*100:.2f}%)")
    print(f"mean : {df['loading'].mean():.2f} passengers")
    print(f"max : {df['loading'].max():.0f} passengers")
    print(f"negative : {(df['loading'] < 0).sum():,} records (will be filtered)")

    # date range
    if 'gps_datetime' in df.columns:
        if not pd.api.types.is_datetime64_any_dtype(df['gps_datetime']):
            df['gps_datetime'] = pd.to_datetime(df['gps_datetime'], errors='coerce')
        print(f"\ndate range: {df['gps_datetime'].min()} → {df['gps_datetime'].max()}")

    print(f"\nvalidation complete")
    return True

In [13]:
# --- clean and prepare data ---

def prepare_data(df):
    """
    cleans and prepares the raw dataframe for feature engineering.
    - converts timestamps to datetime
    - sorts by route, direction, trip and stop sequence (important for lag features later)
    - removes negative loading values and nulls in critical columns
    """
    print(f"\n{'='*60}")
    print("Preparing data")
    print(f"{'='*60}")

    df = df.copy()
    n_original = len(df)

    # convert timestamp columns to datetime
    print("Converting timestamps...")
    for col in ['stop_time', 'start_trip', 'end_trip']:
        if col in df.columns and not pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = pd.to_datetime(df[col], errors='coerce')

    # sort by route → direction → trip → stop sequence
    # this ordering is critical: lag features depend on sequential stop order
    print("sorting by route, direction, trip and stop sequence...")
    df = df.sort_values(['route_short_name', 'direction_id', 'start_trip', 'pt_sequence'])
    df = df.reset_index(drop=True)

    # remove records with negative passenger counts (physically impossible)
    if 'loading' in df.columns:
        n_before = len(df)
        df = df[df['loading'] >= 0]
        removed = n_before - len(df)
        if removed > 0:
            print(f"removed {removed:,} records with negative loading")

    # remove records with nulls in critical columns
    critical = [c for c in ['loading', 'route_short_name', 'stop_id'] if c in df.columns]
    n_before = len(df)
    df = df.dropna(subset=critical)
    removed = n_before - len(df)
    if removed > 0:
        print(f"removed {removed:,} records with null values in critical columns")

    print(f"\nOriginal records : {n_original:,}")
    print(f"Final records : {len(df):,}")
    print(f"Removed total : {n_original - len(df):,} ({(n_original - len(df))/n_original*100:.2f}%)")
    print(f"Memory usage : {df.memory_usage(deep=True).sum() / 1024**2:.2f} mb")

    return df

In [14]:
# --- save data to drive ---

def save_data(df, year, month, fmt='parquet'):
    """
    saves the processed dataframe to the dataset folder in drive.
    format options: 'parquet' (recommended), 'csv', 'pickle'.
    parquet is preferred because it preserves data types and compresses well.
    """
    os.makedirs(SAVE_PATH, exist_ok=True)
    month_name = calendar.month_name[month].lower()
    filepath = f"{SAVE_PATH}/sunt_od_{year}_{month:02d}_{month_name}.{fmt}"

    print(f"\nSaving to {filepath}...")

    if fmt == 'parquet':
        df.to_parquet(filepath, index=False)
    elif fmt == 'csv':
        df.to_csv(filepath, index=False)
    elif fmt == 'pickle':
        df.to_pickle(filepath)
    else:
        print(f"Unknown format: {fmt}")
        return None

    size_mb = os.path.getsize(filepath) / 1024**2
    print(f"saved successfully ({size_mb:.2f} mb) → {filepath}")
    return filepath

In [17]:
# --- main execution ---
# downloads all months defined in MONTHS_TO_DOWNLOAD,
# combines them into a single dataframe, and saves it.
# if you add a new month later, just add it to MONTHS_TO_DOWNLOAD
# and re-run — the output file will contain all months combined.

import os

all_dfs = []

for (year, month) in MONTHS_TO_DOWNLOAD:
    df_month = download_month(year, month)

    if df_month is None:
        print(f"Skipping {calendar.month_name[month]} {year} — no data downloaded")
        continue

    if VALIDATE_DATA:
        validate_data(df_month)

    df_month = prepare_data(df_month)
    all_dfs.append(df_month)

# combine all months into one dataframe
if all_dfs:
    df = pd.concat(all_dfs, ignore_index=True)

    # re-sort after combining months to preserve sequential order
    df = df.sort_values(['route_short_name', 'direction_id', 'start_trip', 'pt_sequence'])
    df = df.reset_index(drop=True)

    print(f"\n{'='*60}")
    print(f"Combined dataset: {len(df):,} records from {len(all_dfs)} month(s)")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} mb")
    print(f"{'='*60}")

    # save combined file
    months_str = "_".join([f"{y}{m:02d}" for (y, m) in MONTHS_TO_DOWNLOAD])

    # save with a descriptive combined filename
    combined_path = f"{SAVE_PATH}/sunt_od_{months_str}.parquet"
    df.to_parquet(combined_path, index=False)
    size_mb = os.path.getsize(combined_path) / 1024**2
    print(f"File saved → {combined_path} ({size_mb:.2f} mb)")

else:
    print("No data was downloaded. check your month configuration and internet connection.")


Days in month: 31


downloading: 100%|██████████| 31/31 [00:10<00:00,  2.95day/s]



Days downloaded successfully: 31/31
total records: 19,517,947
memory usage: 8478.02 mb

Data validation
  ok       loading              - passenger count on bus (critical)
  ok       n-boardings          - number of boardings at stop
  ok       n-alighting          - number of alightings at stop
  ok       route_short_name     - bus line
  ok       stop_id              - stop identifier
  ok       direction_id         - trip direction
  ok       pt_sequence          - stop sequence within trip
  ok       stop_time            - timestamp

loading (passenger count) stats:
valid values : 19,513,956
nan values : 3,991 (0.02%)
mean : 20.34 passengers
max : 407 passengers
negative : 0 records (will be filtered)

validation complete

Preparing data
Converting timestamps...
sorting by route, direction, trip and stop sequence...
removed 3,991 records with negative loading

Original records : 19,517,947
Final records : 19,513,956
Removed total : 3,991 (0.02%)
Memory usage : 4828.72 mb

Days in 

downloading: 100%|██████████| 30/30 [00:06<00:00,  4.30day/s]



Days downloaded successfully: 5/30
total records: 1,836,504
memory usage: 797.68 mb

Data validation
  ok       loading              - passenger count on bus (critical)
  ok       n-boardings          - number of boardings at stop
  ok       n-alighting          - number of alightings at stop
  ok       route_short_name     - bus line
  ok       stop_id              - stop identifier
  ok       direction_id         - trip direction
  ok       pt_sequence          - stop sequence within trip
  ok       stop_time            - timestamp

loading (passenger count) stats:
valid values : 1,836,080
nan values : 424 (0.02%)
mean : 16.20 passengers
max : 311 passengers
negative : 0 records (will be filtered)

validation complete

Preparing data
Converting timestamps...
sorting by route, direction, trip and stop sequence...
removed 424 records with negative loading

Original records : 1,836,504
Final records : 1,836,080
Removed total : 424 (0.02%)
Memory usage : 454.29 mb

Days in month: 30


downloading: 100%|██████████| 30/30 [00:07<00:00,  4.06day/s]



Days downloaded successfully: 20/30
total records: 12,962,583
memory usage: 3110.74 mb

Data validation
  ok       loading              - passenger count on bus (critical)
  ok       n-boardings          - number of boardings at stop
  ok       n-alighting          - number of alightings at stop
  ok       route_short_name     - bus line
  ok       stop_id              - stop identifier
  ok       direction_id         - trip direction
  ok       pt_sequence          - stop sequence within trip
  ok       stop_time            - timestamp

loading (passenger count) stats:
valid values : 12,962,583
nan values : 0 (0.00%)
mean : 19.49 passengers
max : 233 passengers
negative : 2 records (will be filtered)

validation complete

Preparing data
Converting timestamps...
sorting by route, direction, trip and stop sequence...
removed 2 records with negative loading

Original records : 12,962,583
Final records : 12,962,581
Removed total : 2 (0.00%)
Memory usage : 3209.64 mb

Days in month: 31


downloading: 100%|██████████| 31/31 [00:07<00:00,  4.27day/s]



Days downloaded successfully: 11/31
total records: 7,543,248
memory usage: 1810.06 mb

Data validation
  ok       loading              - passenger count on bus (critical)
  ok       n-boardings          - number of boardings at stop
  ok       n-alighting          - number of alightings at stop
  ok       route_short_name     - bus line
  ok       stop_id              - stop identifier
  ok       direction_id         - trip direction
  ok       pt_sequence          - stop sequence within trip
  ok       stop_time            - timestamp

loading (passenger count) stats:
valid values : 7,543,248
nan values : 0 (0.00%)
mean : 20.30 passengers
max : 438 passengers
negative : 31 records (will be filtered)

validation complete

Preparing data
Converting timestamps...
sorting by route, direction, trip and stop sequence...
removed 31 records with negative loading

Original records : 7,543,248
Final records : 7,543,217
Removed total : 31 (0.00%)
Memory usage : 1867.61 mb

Combined dataset: 41,

In [ ]:
# --- push changes to github ---
!git add .
!git commit -m "first update dataloader with 4 months"
!git push origin floppy

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
